# Autoencoder convolucional para eliminación de ruido

Ejemplo con MNIST y `tf.keras`.

## Dependencias

En un entorno nuevo: `%pip install tensorflow numpy matplotlib`. La primera carga descarga MNIST.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEED = 2026
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
x_train = x_train[..., None].astype('float32') / 255.0
x_test = x_test[..., None].astype('float32') / 255.0
rng = np.random.default_rng(SEED)
noise = 0.35
x_train_noisy = np.clip(x_train + noise * rng.normal(size=x_train.shape), 0, 1).astype('float32')
x_test_noisy = np.clip(x_test + noise * rng.normal(size=x_test.shape), 0, 1).astype('float32')

In [ ]:
inputs = keras.Input(shape=(28, 28, 1))
x = keras.layers.Conv2D(32, 3, activation='relu', padding='same', strides=2)(inputs)
x = keras.layers.Conv2D(64, 3, activation='relu', padding='same', strides=2)(x)
x = keras.layers.Conv2DTranspose(64, 3, activation='relu', padding='same', strides=2)(x)
x = keras.layers.Conv2DTranspose(32, 3, activation='relu', padding='same', strides=2)(x)
outputs = keras.layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)
autoencoder = keras.Model(inputs, outputs, name='denoising_autoencoder')
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.summary()

In [ ]:
history = autoencoder.fit(
    x_train_noisy, x_train,
    validation_split=0.1, epochs=8, batch_size=128, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)],
)

In [ ]:
recon = autoencoder.predict(x_test_noisy[:10], verbose=0)
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
for i in range(10):
    axes[0, i].imshow(x_test_noisy[i, ..., 0], cmap='gray')
    axes[1, i].imshow(recon[i, ..., 0], cmap='gray')
    axes[2, i].imshow(x_test[i, ..., 0], cmap='gray')
    for row in range(3): axes[row, i].axis('off')
axes[0, 0].set_ylabel('Ruidosa'); axes[1, 0].set_ylabel('Reconstruida'); axes[2, 0].set_ylabel('Objetivo')
plt.tight_layout(); plt.show()

## Distribución del error de reconstrucción

In [ ]:
recon_all = autoencoder.predict(x_test_noisy, batch_size=256, verbose=0)
mse = np.mean((x_test - recon_all) ** 2, axis=(1, 2, 3))
plt.hist(mse, bins=50)
plt.xlabel('MSE por imagen'); plt.ylabel('Frecuencia'); plt.grid(alpha=0.3)
plt.show()

## Trabajo propuesto

Entrene solo con dígitos de una clase y evalúe detección de otras clases. Calibre el umbral con validación y reporte PR-AUC; no lo elija mirando prueba.